# 01 遅延評価と実行計画

**目的**: DataFrame API は「処理を実行する API」ではなく「実行計画を組み立てる API」だと、手を動かして確かめる（60分）。

終わったときに説明できるようになること:

- transformation と action の違い。計算はいつ走るのか
- `explain()` の読み方。書いたコードと、実際に実行される計画がどう違うか
- 列の刈り込み、述語プッシュダウン、パーティションの刈り込み
- 最適化が効かなくなる書き方（UDF）と、`cache()` の効きどころ

**進め方**: 各実験で、先に予想を書く → 実行する → 観察を書く。`# TODO` のセルは `...` を自分で埋める。直後のセルが ✅ / ❌ で確かめてくれる。

In [ ]:
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType

from labkit import DATA, check_plan, get_spark, large, materialize, observe, same_optimized_plan, timed

spark = get_spark("01_lazy_evaluation", {"spark.sql.adaptive.enabled": "false"})

orders = large(spark, "orders")
customers = large(spark, "customers")

`spark.sql.adaptive.enabled`（AQE）を切っているのは、Spark が実行中に計画を書き換える機能を止めて、素の挙動を見るため。AQE は `03_shuffle` の最後で ON にする。

---

## 実験1: transformation では何も実行されない

✏️ **予想: 500万行の注文を filter して groupBy する DataFrame を「定義する」のに何秒かかる？ そのあと count() を呼ぶと？**

（ここに書く）

In [ ]:
# TODO 1-1: completed な注文だけを残し、顧客ごとの注文数を数える DataFrame を作る
#   使うもの: orders.filter(...) / .groupBy("customer_id") / .agg(F.count("*").alias("n_orders"))
#   ここではまだ show() や count() を呼ばない
with timed("定義するだけ"):
    per_customer = ...

In [ ]:
with observe(spark, "action を呼ぶ"):
    print(f"{per_customer.count():,} 顧客")

✏️ **観察: 定義にかかった時間と、count() にかかった時間は？ Spark UI の Jobs タブにジョブが現れたのは、どのセルを実行したとき？**

（ここに書く）

---

## 実験2: explain で計画を読む

`explain(mode="extended")` は、書いたコードが実行されるまでの4段階を表示する。

| 段階 | 中身 |
|---|---|
| Parsed Logical Plan | 書いたとおりの処理の並び |
| Analyzed Logical Plan | 列名や型を解決したもの |
| Optimized Logical Plan | Spark（Catalyst）が書き換えた後の処理の並び |
| Physical Plan | 実際にどう実行するか（どの結合方式か、どこで shuffle するか） |

計画は木になっていて、**下から上に**読む。いちばん下の `FileScan` がファイルの読み込みで、データは上に向かって流れる。

In [ ]:
per_customer.explain(mode="extended")

✏️ **観察: Parsed と Optimized を見比べる。Optimized にだけ現れた演算子は何？ Physical Plan の FileScan の ReadSchema には、orders の5列のうちどの列が載っている？**

（ここに書く）

In [ ]:
# TODO 2-1: 「completed な注文に、顧客の segment を付けたもの」を2通りの順番で書く
#   a: orders を先に filter してから、customers と customer_id で join する
#   b: orders と customers を join してから、filter する
#   どちらも最後に .select("order_id", "segment") する
a = ...
b = ...

In [ ]:
same_optimized_plan(a, b)
b.explain()

✏️ **観察: b では join の後に filter を書いた。Physical Plan では、Filter は join の前と後のどちらにある？ 書く順番は性能に影響した？**

（ここに書く）

---

## 実験3: 列の刈り込みとプッシュダウン

Physical Plan の `FileScan parquet` の行には、ファイルの読み方が書いてある。

| 項目 | 意味 |
|---|---|
| `ReadSchema` | 実際に読む列。Parquet は列ごとに保存されているので、使わない列は読まずに済む（列の刈り込み） |
| `PushedFilters` | Parquet の読み取り側に渡した条件。ファイル内の統計（最小値・最大値）を見て、条件に合う行がないブロックを読み飛ばせる（述語プッシュダウン） |
| `PartitionFilters` | ディレクトリ名（`order_month=2026-03` など）で絞る条件。該当しないディレクトリは開きもしない（パーティションの刈り込み） |

In [ ]:
# TODO 3-1: orders から、status が 'completed' の行の order_id と customer_id だけを取り出す
slim = ...

In [ ]:
slim.explain()
check_plan(slim, has=["EqualTo(status,completed)"], lacks=["ordered_at", "coupon_code"])

次はパーティションの刈り込み。`data/large/orders_by_month` は、同じ注文データを `order_month=2025-01/` のような月ごとのディレクトリに分けて保存してある。

In [ ]:
# TODO 3-2: orders_by_month から 2026年3月の注文だけを取り出す。order_month 列（"2026-03" という文字列）で絞る
by_month = large(spark, "orders_by_month")
march = ...

In [ ]:
check_plan(march, has=["PartitionFilters: [isnotnull(order_month"])

# 比較用: 同じ「2026年3月」を、ディレクトリの列ではなく ordered_at の範囲で絞ったもの
march_by_time = by_month.filter((F.col("ordered_at") >= "2026-03-01") & (F.col("ordered_at") < "2026-04-01"))
march_by_time.explain(mode="formatted")

with observe(spark, "order_month で絞る"):
    materialize(march)
with observe(spark, "ordered_at で絞る"):
    materialize(march_by_time)

✏️ **観察: 2つの「ファイル読み込み」の行を比べる。開いたファイル数と読んだ行数はそれぞれいくつ？ ordered_at で絞ったほうの計画では、PushedFilters に条件が載っている。それなのに読んだ行数は減った？**

（ここに書く）

`PushedFilters` に載るのは「Spark が Parquet の読み取り側に条件を渡した」というだけで、実際に読み飛ばせたかどうかは別の話になる。読み飛ばしにはファイル内の統計（列の最小値・最大値）が要るが、Spark が日時を Parquet に書くときの既定の形式（INT96）には、その統計が付かない。

同じデータを、統計が付く形式（TIMESTAMP_MICROS）で書き直して比べてみる。

In [ ]:
micros_path = str(DATA / "tmp" / "orders_by_month_micros")
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
by_month.repartition("order_month").write.mode("overwrite").partitionBy("order_month").parquet(micros_path)
spark.conf.set("spark.sql.parquet.outputTimestampType", "INT96")  # 既定値に戻す

march_micros = (
    spark.read.parquet(micros_path)
    .filter((F.col("ordered_at") >= "2026-03-01") & (F.col("ordered_at") < "2026-04-01"))
)
with observe(spark, "ordered_at で絞る（TIMESTAMP_MICROS）"):
    materialize(march_micros)

✏️ **観察: 3つの方法（order_month で絞る / ordered_at・INT96 / ordered_at・TIMESTAMP_MICROS）で、開いたファイル数と読んだ行数はどう違った？**

（ここに書く）

ここから言えること:

- **ディレクトリによる刈り込み**は、該当しないファイルを開きもしない。いちばん強い
- **統計による読み飛ばし**は、ファイルは開くが、中身のブロックを読まずに済ませる。型やファイルの書き方によって、効くときと効かないときがある
- 計画に書いてあることと、実際に起きたことは違う場合がある。読んだ行数を見て確かめる

「よく絞る列でディレクトリを分ける」という設計はここから来ている。ただし分けすぎると小さいファイルが大量にできる（`02_partitions` で扱う）。Delta Lake の OPTIMIZE や Liquid Clustering は、統計による読み飛ばしが効きやすいようにデータを並べ直す仕組みだ（`databricks/` で扱う）。

---

## 実験4: Python の UDF を挟むと最適化が効かない

Spark が最適化できるのは、中身が分かっている処理だけ。Python の関数（UDF）は Spark から見るとブラックボックスなので、条件を Parquet に渡せないし、全行を Python のプロセスに送って1行ずつ評価することになる。

In [ ]:
is_completed = F.udf(lambda s: s == "completed", BooleanType())
slim_udf = orders.filter(is_completed("status")).select("order_id", "customer_id")

slim_udf.explain()
with observe(spark, "組み込みの比較（slim）"):
    materialize(slim)
with observe(spark, "Python UDF（slim_udf）"):
    materialize(slim_udf)

✏️ **観察: slim_udf の計画で、PushedFilters はどうなった？ slim にはなかった演算子は？ 実行時間は何倍違う？**

（ここに書く）

業務で動かしている Spark のジョブ（AWS Glue など）に UDF があれば、組み込み関数（`pyspark.sql.functions`）に置き換えられないかが最初の改善候補になる。

---

## 実験5: action を呼ぶたびに最初から計算される

In [ ]:
# TODO 5-1: per_customer.count() を2回実行する。それぞれ observe で囲む
with observe(spark, "1回目"):
    ...
with observe(spark, "2回目"):
    ...

✏️ **観察: 2回目もファイルを読んでいる？（「ファイル読み込み」の行を見る）**

（ここに書く）

In [ ]:
# TODO 5-2: per_customer.cache() を呼んでから、もう一度 count() を2回実行する
per_customer.cache()
with observe(spark, "cache 後 1回目"):
    ...
with observe(spark, "cache 後 2回目"):
    ...

In [ ]:
per_customer.filter("n_orders >= 3").explain()

✏️ **観察: cache 後の1回目と2回目で、読んだファイル数と時間はどう変わった？ キャッシュが作られたのは cache() を呼んだとき？ それとも？ 上の計画で FileScan の代わりに現れた演算子は？ Spark UI の Storage タブには何が出ている？**

（ここに書く）

In [ ]:
per_customer.unpersist()
shutil.rmtree(DATA / "tmp", ignore_errors=True)  # 実験3で書いた一時データを消す

`cache()` も transformation と同じで、呼んだ時点では何も起きない。次の action で計算した結果がメモリに残り、その次から使われる。同じ DataFrame を何度も使うときだけ意味があり、1回しか使わないなら、メモリを使うぶん損になる。

---

## まとめ

✏️ **このノートブックで分かったことを、自分の言葉で3行にまとめる**

（ここに書く）

### 説明できるか確かめる

答えを見ずに、声に出して説明してみる。詰まったところが、まだ分かっていないところ。

1. transformation と action は何が違う？ Spark UI のどこを見れば区別できる？
2. `df.filter(...).select(...)` と `df.select(...).filter(...)` で性能が変わらないのはなぜ？ 変わってしまうのはどんなとき？
3. `PushedFilters` と `PartitionFilters` は何が違う？
4. `cache()` が逆効果になるのはどんなとき？

In [ ]:
spark.stop()